# geocoder-service — exemples

Nécessite la stack Docker démarrée (`docker compose --env-file .env up -d`) et les dépendances installées (`uv sync`).

## 1. Recherche simple

`geocode()` renvoie les résultats triés par score de confiance (0-100) décroissant.

In [ ]:
from geocoder_service.search import geocode

for hit in geocode("Av. de Thonex 30", limit=3):
    print(f"{hit['score']:6.2f}  {hit['adresse']:30s} {hit['commune']}")

## 2. Abréviations de types de voie

Les abréviations (`ch.`, `av.`, `bd`, `rte`, ...) sont extraites automatiquement des données SITG elles-mêmes (`TYPABR`/`TYVOIE`), pas d'une liste écrite à la main.

In [ ]:
for q in ["Ch de la Bassonette, 9", "Rte de Sauverny", "Bd James-Fazy 2"]:
    hits = geocode(q, limit=1)
    top = hits[0] if hits else None
    print(f"{q:25s} -> {top['score']:6.2f}  {top['adresse']}" if top else f"{q:25s} -> aucun résultat")

## 3. Initiales de prénoms composés (et ambiguïté)

`"j-d"` est aussi dérivé automatiquement du champ `LIANT` (ex. "Jacob-Daniel" → "j-d"). Certaines initiales sont ambiguës (ex. "j-d" = "Jean-Daniel" *ou* "Jacob-Daniel") : toutes les interprétations sont essayées, et le reste de l'adresse (ici "Maillard") départage.

In [ ]:
for hit in geocode("Av. J-D Maillard, 7", limit=3):
    print(f"{hit['score']:6.2f}  {hit['adresse']:35s} {hit['commune']}")

## 4. Numéros de rue : suffixes "bis"/"ter" et plages

- `"7b"` est reconnu comme équivalent de `"7bis"` (vérifié sur les adresses réelles en BIS/TER du jeu de données).
- Une plage en fin de requête (ex. `"51-53"`) accepte l'une ou l'autre des deux bornes : les deux existent souvent comme adresses distinctes.

In [ ]:
for q in ["Rue Dizerens, 7b", "Chemin de la Montagne 51-53", "Avenue de Châtelaine 60-62"]:
    for hit in geocode(q, limit=2):
        print(f"{q:32s} -> {hit['score']:6.2f}  {hit['adresse']}")
    print()

## 5. Robustesse au mauvais type de voie

L'utilisateur écrit "rue" mais la vraie adresse est un "chemin". Le mot de type de voie est neutralisé pour le classement Meilisearch (`stopWords`), donc il ne fait pas gagner un mauvais candidat juste parce qu'il partage un mot très fréquent.

In [ ]:
for hit in geocode("rue du Pré-du-Couvent 1bis", limit=2):
    print(f"{hit['score']:6.2f}  {hit['adresse']:35s} {hit['typeVoie']}")

## 6. Via l'API HTTP

Utile pour d'autres langages/services (ex. `sitg_geocode.py` pourrait pointer ici) : mêmes résultats, via `GET /search`.

In [ ]:
import json
import urllib.parse
import urllib.request

params = urllib.parse.urlencode({"q": "Av. de Thonex 30", "limit": 3})
with urllib.request.urlopen(f"http://localhost:8000/search?{params}") as resp:
    data = json.load(resp)
data

## 7. Comparaison sur le jeu de test complet

Reproduit `compare.py` : sur les 107 adresses de `tests/test_adresses.csv`, combien passent un seuil de score de 95 ?

In [ ]:
import csv
from pathlib import Path

with open(Path("tests") / "test_adresses.csv", encoding="utf-8") as f:
    addresses = [row[0] for row in csv.reader(f)][1:]

n_ge95 = 0
below = []
for adr in addresses:
    hits = geocode(adr, limit=1)
    top = hits[0] if hits else None
    if top and top["score"] >= 95:
        n_ge95 += 1
    else:
        below.append((top["score"] if top else None, adr, top["adresse"] if top else None))

print(f"{n_ge95}/{len(addresses)} avec un score >= 95")
below[:10]